# Text-to-SVG: QLoRA Fine-Tuning on Qwen2.5-Coder-1.5B

**NYU Deep Learning Spring 2026 — Kaggle Competition**

**Environment:** Google Colab Pro with A100 GPU + Google Drive

### Setup Instructions
1. Upload `train.csv` and `test.csv` to your Google Drive under `MyDrive/svg-competition/`
2. Set Runtime → Change runtime type → **GPU** (A100 if available)
3. Run all cells in order
4. Adapter weights will be saved to `MyDrive/svg-competition/svg-lora-adapter/`

## 1. Mount Google Drive & Install Dependencies

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

# ── Create project folder if it doesn't exist ──
import os
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')
print(f'Contents: {os.listdir(PROJECT_DIR)}')

In [ ]:
# ── Install dependencies ──
!pip install -q transformers accelerate peft bitsandbytes datasets trl cairosvg lxml pandas

In [ ]:
import os
import re
import time
import random
import json
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# ════════════════════════════════════════════════════════════
# ALL PATHS USE GOOGLE DRIVE
# Put train.csv and test.csv in: MyDrive/svg-competition/
# ════════════════════════════════════════════════════════════

PROJECT_DIR = '/content/drive/MyDrive/svg-competition'

CONFIG = {
    # ── Model ──
    'model_name': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'max_seq_length': 1536,              # Covers median SVG (~500-600 tokens) + prompt overhead

    # ── LoRA ──
    'lora_r': 32,                        # 2x starter (16), good expressiveness vs speed tradeoff
    'lora_alpha': 64,                    # alpha/r = 2.0 scaling
    'lora_dropout': 0.05,

    # ── Training ──
    'learning_rate': 2e-4,
    'num_train_epochs': 2,               # 2 full passes over training data
    'per_device_train_batch_size': 12,     # Larger batch → fewer steps per epoch
    'gradient_accumulation_steps': 1,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'max_grad_norm': 0.3,

    # ── Logging & Saving ──
    'logging_steps': 25,
    'save_steps': 500,
    'eval_steps': 500,
    'save_total_limit': 3,
    'output_dir': '/content/svg-lora-checkpoints',        # Local (faster I/O during training)

    # ── Data (Google Drive paths) ──
    'train_csv': f'{PROJECT_DIR}/train.csv',
    'test_csv': f'{PROJECT_DIR}/test.csv',
    'eval_fraction': 0.02,
    'max_svg_len': 8000,
    'min_svg_len': 80,

    # ── Adapter Output (Google Drive — persists after session) ──
    'adapter_save_dir': f'{PROJECT_DIR}/svg-lora-adapter',
}

# Verify data files exist
for key in ['train_csv', 'test_csv']:
    path = CONFIG[key]
    exists = os.path.exists(path)
    print(f'{key}: {path} — {"FOUND" if exists else "NOT FOUND ⚠️"}')

print(f'\nFull config:')
print(json.dumps(CONFIG, indent=2))

## 3. Load & Clean Training Data

In [ ]:
df = pd.read_csv(CONFIG['train_csv'])
print(f'Raw dataset: {len(df)} rows')
print(f'Columns: {list(df.columns)}')
df.head(2)

In [ ]:
# ── Allowed SVG tags (from competition rules) ──
ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter'
}

def validate_svg(svg_text):
    """Check if SVG is valid XML, uses only allowed tags, and meets size constraints."""
    if not svg_text or not isinstance(svg_text, str):
        return False, 'empty'
    if not svg_text.strip().startswith('<svg'):
        return False, 'no_svg_root'
    if len(svg_text) > 8000:
        return False, 'too_long'
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False, 'parse_error'

    path_count = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag not in ALLOWED_TAGS:
            return False, f'disallowed_tag:{tag}'
        if tag == 'path':
            path_count += 1
    if path_count > 256:
        return False, 'too_many_paths'
    return True, 'ok'


def normalize_svg(svg_text):
    """Light normalization: clean whitespace."""
    svg_text = re.sub(r'\s+', ' ', svg_text).strip()
    return svg_text


# ── Filter and clean ──
valid_rows = []
reject_reasons = Counter()

for _, row in df.iterrows():
    svg = str(row['svg']).strip()
    prompt = str(row['prompt']).strip()

    if len(svg) < CONFIG['min_svg_len']:
        reject_reasons['too_short'] += 1
        continue
    if len(svg) > CONFIG['max_svg_len']:
        reject_reasons['filtered_long'] += 1
        continue
    if not prompt or len(prompt) < 5:
        reject_reasons['bad_prompt'] += 1
        continue

    is_valid, reason = validate_svg(svg)
    if not is_valid:
        reject_reasons[reason] += 1
        continue

    valid_rows.append({
        'id': row['id'],
        'prompt': prompt,
        'svg': normalize_svg(svg),
    })

clean_df = pd.DataFrame(valid_rows)
print(f'Clean dataset: {len(clean_df)} / {len(df)} rows ({100*len(clean_df)/len(df):.1f}%)')
print(f'\nRejection reasons:')
for reason, count in reject_reasons.most_common():
    print(f'  {reason}: {count}')
print(f'\nSVG length stats (clean):')
print(clean_df['svg'].str.len().describe())

In [ ]:
# ── Train/val split ──
clean_df = clean_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_eval = max(100, int(len(clean_df) * CONFIG['eval_fraction']))

eval_df = clean_df.iloc[:n_eval]
train_df = clean_df.iloc[n_eval:]

print(f'Train: {len(train_df)} rows')
print(f'Eval:  {len(eval_df)} rows')

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
eval_dataset = Dataset.from_pandas(eval_df, preserve_index=False)

## 4. Format Data for SFT

In [ ]:
SYSTEM_PROMPT = (
    'You are an expert SVG code generator. Given a description, generate clean, valid SVG code. '
    'Output only the SVG code with a single root <svg> element. '
    "Use xmlns='http://www.w3.org/2000/svg', width='256', height='256', viewBox='0 0 256 256'. "
    'Keep the SVG compact and under 8000 characters. Use at most 256 path elements.'
)

def format_chat(example):
    text = (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f"{example['prompt']}<|im_end|>\n"
        '<|im_start|>assistant\n'
        f"{example['svg']}<|im_end|>"
    )
    return {'text': text}

train_formatted = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
eval_formatted = eval_dataset.map(format_chat, remove_columns=eval_dataset.column_names)

print('Sample formatted text (first 600 chars):')
print(train_formatted[0]['text'][:600])
print(f'\n... (total length: {len(train_formatted[0]["text"])} chars)')

## 5. Load Model with QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# ── 4-bit quantization ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── Load tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Load model ──
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

print(f'Model loaded: {CONFIG["model_name"]}')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
# ── Apply LoRA ──
lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_strategy='steps',
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    gradient_checkpointing=True,
    seed=SEED,
    dataloader_pin_memory=True,
    # SFT-specific
    max_length=CONFIG['max_seq_length'],
    packing=False,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    args=sft_config,
)

effective_batch = CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
steps_per_epoch = len(train_formatted) // effective_batch
total_steps = steps_per_epoch * CONFIG['num_train_epochs']
print(f'Effective batch size: {effective_batch}')
print(f'Steps per epoch: ~{steps_per_epoch}')
print(f'Total training steps: ~{total_steps}')

In [ ]:
# ── TRAIN ──
t0 = time.time()
train_result = trainer.train()
elapsed = (time.time() - t0) / 60

print(f'\nTraining complete in {elapsed:.1f} minutes')
print(f'Final train loss: {train_result.training_loss:.4f}')

## 7. Save Adapter to Google Drive

This saves directly to Drive so it persists after the Colab session ends.

In [ ]:
# ── Save LoRA adapter to Google Drive ──
adapter_dir = CONFIG['adapter_save_dir']
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Show saved files
adapter_files = os.listdir(adapter_dir)
total_size = sum(
    os.path.getsize(os.path.join(adapter_dir, f))
    for f in adapter_files
    if os.path.isfile(os.path.join(adapter_dir, f))
)
print(f'Saved adapter to Google Drive: {adapter_dir}')
print(f'Files: {adapter_files}')
print(f'Total size: {total_size / 1e6:.1f} MB')
print(f'\nThis persists even if Colab disconnects!')

In [ ]:
# ── Also save the base model to Drive for Kaggle offline use ──
BASE_MODEL_SAVE_DIR = f'{PROJECT_DIR}/qwen2.5-coder-1.5b-instruct'

if not os.path.exists(BASE_MODEL_SAVE_DIR):
    print('Saving base model to Google Drive (one-time, ~3 GB)...')
    print('This is needed for Kaggle inference (no internet).')

    from transformers import AutoModelForCausalLM, AutoTokenizer

    # Download full-precision model for Kaggle
    base_tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
    base_model = AutoModelForCausalLM.from_pretrained(
        CONFIG['model_name'],
        torch_dtype=torch.float16,
    )
    base_model.save_pretrained(BASE_MODEL_SAVE_DIR)
    base_tokenizer.save_pretrained(BASE_MODEL_SAVE_DIR)

    print(f'Base model saved to: {BASE_MODEL_SAVE_DIR}')
else:
    print(f'Base model already saved at: {BASE_MODEL_SAVE_DIR}')

## 8. Quick Validation — Generate Sample SVGs

In [ ]:
model.eval()

SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ''

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith('svg')
    except ET.ParseError:
        return False

def generate_svg(prompt, max_new_tokens=1536, temperature=0.6, top_p=0.9):
    chat_text = (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{prompt}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    return extract_svg(decoded)

# ── Test a few prompts ──
test_prompts = [
    'a simple blue bird icon',
    'a red heart shape on white background',
    'a green tree with brown trunk',
]

for p in test_prompts:
    svg = generate_svg(p)
    valid = is_valid_svg(svg)
    print(f'Prompt: {p}')
    print(f'  Valid: {valid} | Length: {len(svg)} chars')
    if valid:
        print(f'  Preview: {svg[:150]}...')
    print()

In [ ]:
# ── Render a sample SVG ──
try:
    import cairosvg
    from IPython.display import Image, display

    for p in ['a yellow star with five points', 'a simple red car']:
        sample_svg = generate_svg(p)
        if is_valid_svg(sample_svg):
            png_bytes = cairosvg.svg2png(
                bytestring=sample_svg.encode('utf-8'),
                output_width=256, output_height=256
            )
            print(f'\nPrompt: {p} ({len(sample_svg)} chars)')
            display(Image(data=png_bytes))
        else:
            print(f'Invalid SVG for: {p}')
except ImportError:
    print('cairosvg not installed — skip visual preview')

## 9. Training Summary

In [ ]:
summary = {
    'model': CONFIG['model_name'],
    'lora_r': CONFIG['lora_r'],
    'lora_alpha': CONFIG['lora_alpha'],
    'epochs': CONFIG['num_train_epochs'],
    'effective_batch_size': CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps'],
    'learning_rate': CONFIG['learning_rate'],
    'train_samples': len(train_formatted),
    'eval_samples': len(eval_formatted),
    'final_train_loss': train_result.training_loss,
    'training_time_min': elapsed,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
    'seed': SEED,
}

print('\n=== TRAINING SUMMARY (save for report) ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

# Save summary to Drive
with open(f'{PROJECT_DIR}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'\nSummary saved to: {PROJECT_DIR}/training_summary.json')

In [ ]:
# ── Load test prompts ──
test_df = pd.read_csv(CONFIG['test_csv'])
print(f'Test prompts: {len(test_df)} rows')
print(f'Columns: {list(test_df.columns)}')
test_df.head(3)

In [ ]:
# ── Full validation + post-processing utilities ──

def fix_svg_canvas(svg_text):
    """Ensure SVG has correct 256x256 canvas attributes."""
    try:
        root = ET.fromstring(svg_text)
        ns = 'http://www.w3.org/2000/svg'
        root.set('xmlns', ns)
        root.set('width', '256')
        root.set('height', '256')
        if 'viewBox' not in root.attrib:
            root.set('viewBox', '0 0 256 256')
        fixed = ET.tostring(root, encoding='unicode')
        if 'xmlns' not in fixed:
            fixed = fixed.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
        return fixed
    except:
        return svg_text


def full_validate_svg(svg_text):
    """Full validation matching competition rules (8K chars, 256 paths, allowed tags)."""
    if not svg_text:
        return False
    if len(svg_text) > 8000:
        return False
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False
    if not root.tag.endswith('svg'):
        return False
    path_count = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag not in ALLOWED_TAGS:
            return False
        if tag == 'path':
            path_count += 1
    return path_count <= 256


def fallback_svg(prompt):
    """Minimal valid SVG fallback with color awareness."""
    colors = ['red', 'blue', 'green', 'yellow', 'orange', 'purple', 'black', 'white', 'pink', 'brown', 'gray']
    fill = 'gray'
    for c in colors:
        if c in prompt.lower():
            fill = c
            break
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        f'<rect width="256" height="256" fill="white"/>'
        f'<circle cx="128" cy="128" r="64" fill="{fill}"/>'
        '</svg>'
    )


def generate_svg_with_retry(prompt, retries=2):
    """Generate SVG with retry logic and post-processing."""
    chat_text = (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{prompt}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)

    for attempt in range(retries + 1):
        temp = 0.5 + 0.1 * attempt
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=True,
                temperature=temp,
                top_p=0.9,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
            )
        decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)
        svg = extract_svg(decoded)
        if svg:
            svg = fix_svg_canvas(svg)
            if full_validate_svg(svg):
                return svg

    return fallback_svg(prompt)

print('Inference utilities ready.')

In [ ]:
print(f'Training loss: {train_result.training_loss:.4f}')

In [ ]:
print(f'Model device: {model.device}')
print(f'Model training mode: {model.training}')

In [ ]:
# Test 1: No system prompt
inputs = tokenizer(
    '<|im_start|>user\na red circle<|im_end|>\n<|im_start|>assistant\n',
    return_tensors='pt'
).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print("TEST 1 (no system prompt):")
print(tokenizer.decode(out[0], skip_special_tokens=False)[-400:])

# Test 2: Simple system prompt
inputs2 = tokenizer(
    '<|im_start|>system\nGenerate SVG code.<|im_end|>\n<|im_start|>user\na red circle<|im_end|>\n<|im_start|>assistant\n<svg',
    return_tensors='pt'
).to(model.device)

with torch.no_grad():
    out2 = model.generate(**inputs2, max_new_tokens=256, do_sample=False)
print("\nTEST 2 (short system prompt + svg hint):")
print(tokenizer.decode(out2[0], skip_special_tokens=False)[-400:])

In [ ]:
# Raw generation test — see exactly what comes out
chat_text = (
    '<|im_start|>system\n'
    f'{SYSTEM_PROMPT}<|im_end|>\n'
    '<|im_start|>user\n'
    'a red circle on white background<|im_end|>\n'
    '<|im_start|>assistant\n'
)
inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
    )

raw_output = tokenizer.decode(output_ids[0], skip_special_tokens=False)
print("RAW OUTPUT:")
print(raw_output[-500:])

In [ ]:
# Check 1: Is LoRA actually active?
print("LoRA active layers:")
lora_count = 0
for name, module in model.named_modules():
    if 'lora' in name.lower() and hasattr(module, 'weight'):
        lora_count += 1
print(f"  Total LoRA layers: {lora_count}")

# Check 2: Direct generation without our wrapper
inputs = tokenizer(
    '<|im_start|>system\nGenerate SVG code.<|im_end|>\n'
    '<|im_start|>user\n'
    f'{test_df.iloc[0]["prompt"]}<|im_end|>\n'
    '<|im_start|>assistant\n<svg',
    return_tensors='pt'
).to(model.device)

print(f"\nInput tokens: {inputs['input_ids'].shape[1]}")

t1 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
gen_time = time.time() - t1
new_tokens = out.shape[1] - inputs['input_ids'].shape[1]

print(f"Generated tokens: {new_tokens}")
print(f"Time: {gen_time:.1f}s")
print(f"Tokens/sec: {new_tokens/gen_time:.1f}")
print(f"\nOutput:")
print(tokenizer.decode(out[0], skip_special_tokens=False)[-500:])

In [ ]:
# ── Merge LoRA weights into base model for faster inference ──
model = model.merge_and_unload()
model.eval()
print("LoRA merged into base model — inference should be faster now")

# Quick speed test
inputs = tokenizer(
    '<|im_start|>system\nGenerate SVG code.<|im_end|>\n'
    '<|im_start|>user\na red circle<|im_end|>\n'
    '<|im_start|>assistant\n<svg',
    return_tensors='pt'
).to(model.device)

t1 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
gen_time = time.time() - t1
new_tokens = out.shape[1] - inputs['input_ids'].shape[1]
print(f"Speed: {new_tokens/gen_time:.1f} tokens/sec (was 7.6)")
print(tokenizer.decode(out[0], skip_special_tokens=False)[-300:])

In [ ]:
# See raw output for the first test prompt
prompt = test_df.iloc[0]['prompt']
print(f"Prompt: {prompt}\n")

inputs = tokenizer(
    '<|im_start|>system\nGenerate SVG code.<|im_end|>\n'
    f'<|im_start|>user\n{prompt}<|im_end|>\n'
    '<|im_start|>assistant\n<svg',
    return_tensors='pt'
).to(model.device)

t1 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=768, do_sample=False)
gen_time = time.time() - t1

raw = tokenizer.decode(out[0], skip_special_tokens=False)
# Show just the assistant response
assistant_part = raw.split('<|im_start|>assistant\n')[-1]
print(f"Time: {gen_time:.1f}s")
print(f"Output length: {len(assistant_part)} chars")
print(f"\nFull output:")
print(assistant_part[:1000])

In [ ]:
def generate_svg_fast(prompt):
    chat_text = (
        '<|im_start|>system\n'
        'Generate SVG code.<|im_end|>\n'
        '<|im_start|>user\n'
        f'{prompt}<|im_end|>\n'
        '<|im_start|>assistant\n'
        '<svg'
    )
    inputs = tokenizer(chat_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=768,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.4,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=False)

    assistant_part = decoded.split('<|im_start|>assistant\n')[-1]
    for token in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
        assistant_part = assistant_part.split(token)[0]

    # Try complete SVG extraction
    svg = extract_svg(assistant_part)

    # If no closing tag, force-close
    if not svg and '<svg' in assistant_part:
        start = assistant_part.index('<svg')
        svg_partial = assistant_part[start:].rstrip()

        # Cut at last complete element
        last_self_close = svg_partial.rfind('/>')
        last_end_tag = svg_partial.rfind('</')
        if last_end_tag != -1:
            try:
                end_pos = svg_partial.index('>', last_end_tag) + 1
                svg_partial = svg_partial[:end_pos]
            except ValueError:
                if last_self_close != -1:
                    svg_partial = svg_partial[:last_self_close + 2]
        elif last_self_close != -1:
            svg_partial = svg_partial[:last_self_close + 2]

        if '</svg>' not in svg_partial:
            svg_partial += '</svg>'
        svg = svg_partial

    if svg:
        # Fix canvas without requiring valid XML parse
        ET.register_namespace('', 'http://www.w3.org/2000/svg')

        # Try clean XML fix first
        try:
            root = ET.fromstring(svg)
            root.set('xmlns', 'http://www.w3.org/2000/svg')
            root.set('width', '256')
            root.set('height', '256')
            if 'viewBox' not in root.attrib:
                root.set('viewBox', '0 0 256 256')
            svg = ET.tostring(root, encoding='unicode')
            svg = svg.replace('ns0:', '').replace(':ns0', '')
        except ET.ParseError:
            # XML is broken — do regex-based fixes instead
            svg = re.sub(r'width="[^"]*"', 'width="256"', svg, count=1)
            svg = re.sub(r'height="[^"]*"', 'height="256"', svg, count=1)
            if 'viewBox' not in svg:
                svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
            if 'xmlns' not in svg:
                svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)

        # Only check length and basic structure, not strict XML validity
        if len(svg) <= 8000 and svg.strip().startswith('<svg') and '</svg>' in svg:
            return svg

    return fallback_svg(prompt)

# Test on first 10 prompts
fallbacks = 0
for i in range(10):
    t1 = time.time()
    svg = generate_svg_fast(test_df.iloc[i]['prompt'])
    elapsed = time.time() - t1
    is_fallback = len(svg) < 190
    if is_fallback:
        fallbacks += 1
    print(f'[{i}] {elapsed:.1f}s | len={len(svg)} | fallback={is_fallback} | {test_df.iloc[i]["prompt"][:50]}...')

print(f'\nFallbacks: {fallbacks}/10')

In [ ]:
def fix_svg_canvas(svg_text):
    """Ensure SVG has correct 256x256 canvas, clean namespace prefixes."""
    try:
        # Register namespace to avoid ns0: prefix
        ET.register_namespace('', 'http://www.w3.org/2000/svg')
        root = ET.fromstring(svg_text)
        root.set('xmlns', 'http://www.w3.org/2000/svg')
        root.set('width', '256')
        root.set('height', '256')
        if 'viewBox' not in root.attrib:
            root.set('viewBox', '0 0 256 256')
        fixed = ET.tostring(root, encoding='unicode')
        if 'xmlns' not in fixed:
            fixed = fixed.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
        # Clean any ns0: prefixes that slip through
        fixed = fixed.replace('ns0:', '').replace(':ns0', '')
        return fixed
    except:
        return svg_text

# Quick verify
svg = generate_svg_fast(test_df.iloc[0]['prompt'])
print(f'Clean: {"ns0" not in svg}')
print(svg[:300])

In [ ]:
rows = []
fallback_count = 0
t0 = time.time()

for idx, row in test_df.iterrows():
    prompt = str(row['prompt']).strip()
    t1 = time.time()
    svg = generate_svg_fast(prompt)
    gen_time = time.time() - t1

    is_fallback = len(svg) < 190
    if is_fallback:
        fallback_count += 1

    rows.append({'id': row['id'], 'svg': svg})

    print(f'  [{idx+1}/{len(test_df)}] {gen_time:.1f}s | len={len(svg)} | fallback={is_fallback} | fallbacks={fallback_count}')

elapsed_total = (time.time() - t0) / 60
print(f'\nDone! Generated {len(rows)} SVGs in {elapsed_total:.1f} minutes')
print(f'Fallback count: {fallback_count} / {len(rows)} ({100*fallback_count/len(rows):.1f}%)')

In [ ]:
# ── Build and verify submission ──
sub_df = pd.DataFrame(rows)

# Final validation pass
final_fixed = 0
for idx, row in sub_df.iterrows():
    if not full_validate_svg(row['svg']):
        sub_df.at[idx, 'svg'] = fallback_svg('')
        final_fixed += 1
if final_fixed > 0:
    print(f'Fixed {final_fixed} invalid SVGs in final pass')

# Stats
svg_lengths = sub_df['svg'].str.len()
print(f'\nSVG length stats:')
print(f'  Mean:   {svg_lengths.mean():.0f} chars')
print(f'  Median: {svg_lengths.median():.0f} chars')
print(f'  Max:    {svg_lengths.max():.0f} chars')
print(f'  Over 8K: {(svg_lengths > 8000).sum()}')

# Verify format
assert list(sub_df.columns) == ['id', 'svg'], f'Wrong columns: {list(sub_df.columns)}'
assert len(sub_df) == len(test_df), f'Row count mismatch'
n_valid = sum(full_validate_svg(s) for s in sub_df['svg'])
print(f'\nValid SVGs: {n_valid}/{len(sub_df)} ({100*n_valid/len(sub_df):.1f}%)')

In [ ]:
# ── Save to Google Drive ──
SUBMISSION_PATH = f'{PROJECT_DIR}/submission.csv'
sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f'Submission saved to: {SUBMISSION_PATH}')
print(f'File size: {os.path.getsize(SUBMISSION_PATH) / 1e6:.2f} MB')
print(f'\nDownload this file and upload it to the Kaggle competition page.')

In [ ]:
# ── Download submission directly from Colab ──
from google.colab import files
files.download(SUBMISSION_PATH)

## AI Tooling Disclosure

- **Claude (Anthropic)**: Coding assistance, debugging.